1.1

In [ ]:
import pandas as pd
# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取帖子和评论文件
posts_df = pd.read_csv(data_dir + "posts.csv")
comments_df = pd.read_csv(data_dir + "comments.csv")

# 通过关联字段 "id:ID"（posts）与 "post"（comments）进行内连接，
# 这样只统计在 posts.csv 中存在的帖子对应的评论
merged_df = pd.merge(posts_df, comments_df, left_on="id:ID", right_on="post", how="inner")

# 统计所有 (pi, cj) 对数量
total_pairs = merged_df.shape[0]

# 统计满足条件的 (pi, cj) 对数量，即评论的 nli_label 为 'bart_entailment'
valid_pairs = merged_df[merged_df["nli_label"] == "bart_entailment"].shape[0]

print("所有 (pi, cj) 对的数量：", total_pairs)
print("满足 nli_label 为 'bart_entailment' 的 (pi, cj) 对的数量：", valid_pairs)


1.2

In [ ]:
import pandas as pd
import math

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 直接过滤评论，检查评论的 nli_label 是否为 'bart_entailment'
valid_comments = comments_df[comments_df['nli_label'] == 'bart_entailment']

# 用户给定的变量 n（代表元组中评论数量）
n = 2  # 举例：n=2 表示 (post, comment1, comment2) 的元组

# 按照帖子分组，统计每个帖子的满足条件评论数量
grouped_counts = valid_comments.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1, ..., comment_n) 元组数量为组合数 C(m, n)
tuple_count = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# 统计所有帖子下 (post, comment1, ..., comment_n) 元组的数量（不限制评论是否满足条件）
all_grouped_counts = comments_df.groupby('post').size()
all_tuple_count = sum(math.comb(m, n) for m in all_grouped_counts if m >= n)

print(f"满足条件的 (post, comment1, ..., comment{n}) 元组数量：", tuple_count)
print(f"所有 (post, comment1, ..., comment{n}) 元组数量：", all_tuple_count)


2.1& 2.2

In [ ]:
import pandas as pd
import math

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选满足条件的帖子：workload1_label 为 'workload1_entailment'
valid_posts_ids = posts_df.loc[posts_df['oracle_label'] == 'deepseek_yes', 'id:ID']

# 从评论中筛选出：评论属于满足条件的帖子（去除对 nli_label 的条件）
valid_comments = comments_df[comments_df['post'].isin(valid_posts_ids)]

# 用户给定的变量 n（代表元组中评论数量）
n = 1  # 此时元组形式为 (post, comment1, comment2, comment3)

# 按照帖子分组，统计每个帖子的评论数量（满足条件）
grouped_counts = valid_comments.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
tuple_count = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# 按照帖子分组，统计每个帖子的所有评论数量
all_grouped_counts = comments_df.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
all_tuple_count = sum(math.comb(m, n) for m in all_grouped_counts if m >= n)

print(f"满足条件的 (post, comment1, ..., comment{n}) 元组数量：", tuple_count)
print(f"所有 (post, comment1, ..., comment{n}) 元组数量：", all_tuple_count)


3.1 & 3.2

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "posts.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 2  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df[users_df['profession'] == 'Education']['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users))]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, post1, ..., post{n}) 元组数量：", edu_tuples)


4.1 & 4.2

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "posts.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 2  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users)) &
                     (posts_df['oracle_label'] == 'deepseek_yes')]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, post1, ..., post{n}) 元组数量：", edu_tuples)


5.1 & 5.2

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "comments.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 1  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df[users_df['profession'] == 'Education']['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users)) 
                     ]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, post1, ..., post{n}) 元组数量：", edu_tuples)


6.1 & 6.2 

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "comments.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 2  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users)) &
                     (posts_df['nli_label'] == 'bart_entailment')]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, comment1, ..., comment{n}) 元组数量：", edu_tuples)


7.

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "posts.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 2  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df[users_df['profession'] == 'Education']['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users)) &
                     (posts_df['oracle_label'] == 'deepseek_yes')]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, post1, ..., post{n}) 元组数量：", edu_tuples)


8.

In [ ]:
import pandas as pd
import math

# 定义数据目录和文件路径
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
POSTS_FILE = data_dir + "comments.csv"
USERS_FILE = data_dir + "users_valid_test2.csv"

# 用户给定的变量 n（表示元组中评论数量，即 (user, post1, ..., postₙ) 中除了 user 外有 n 个帖子）
n = 2  # 举例：n = 3

# 读取 posts 和 users 数据
posts_df = pd.read_csv(POSTS_FILE)
users_df = pd.read_csv(USERS_FILE)

# ---------------------------------------
# 统计所有 (user, post1, ..., postₙ) 元组的数量
# ---------------------------------------

# 按照帖子创建者（creator）分组，统计每个用户创建的帖子数量 m
user_posts_counts = posts_df.groupby('creator').size()

# 对于每个用户，如果 m >= n，则该用户形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in user_posts_counts if m >= n)
print(f"所有 (user, post1, ..., post{n}) 元组数量：", total_tuples)

# --------------------------------------------------
# 统计满足条件的 (user, post1, ..., postₙ) 元组的数量：
# 条件：用户的 profession 为 'Education'
#         且该用户的帖子（在 posts.csv 中）中，其 oracle_label 均为 'deepseek_yes'
# --------------------------------------------------

# 1. 筛选出 profession 为 'Education' 的用户，获取其用户ID
education_users = users_df[users_df['profession'] == 'Education']['id:ID']

# 2. 在 posts_df 中筛选出 creator 在 education_users 内的帖子，
#    同时要求帖子属性 oracle_label 为 'deepseek_yes'
edu_posts = posts_df[(posts_df['creator'].isin(education_users)) &
                     (posts_df['nli_label'] == 'bart_entailment')]

# 3. 按照 creator 分组，统计每个用户在 edu_posts 中的帖子数量 m'
edu_user_posts_counts = edu_posts.groupby('creator').size()

# 4. 对于每个用户，如果 m' >= n，则该用户形成的元组数量为组合数 C(m', n)
edu_tuples = sum(math.comb(m, n) for m in edu_user_posts_counts if m >= n)
print(f"满足条件的 (user, post1, ..., post{n}) 元组数量：", edu_tuples)


In [ ]:
# 获取创建帖子最多的用户 ID
max_user_id = user_posts_counts.idxmax()
max_posts = user_posts_counts.max()

print("创建帖子最多用户的ID：", max_user_id)
print("该用户创建的帖子数量：", max_posts)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# 绘制直方图展示分布情况
plt.figure(figsize=(20, 6))
# 设置 bins 为 1 到最大数量+1，这样每个整数对应一个bin
bins = range(1, user_posts_counts.max() + 2)
sns.histplot(user_posts_counts, bins=bins, discrete=True)
plt.xlabel("每个用户创建的帖子数量")
plt.ylabel("用户数量")
plt.title("帖子创建数量分布图")
plt.xticks(bins)
plt.show()

9.

In [ ]:
import pandas as pd

# 读取评论数据和推文数据
# root = 'D:/softwareResource/datasets/workload4/'
# root = 'D:/softwareResource/datasets/IOGS/many_predicates/independent/dataset_1/'
# root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# comments_df = pd.read_csv(root + 'comments_sorted_valid_4.csv')
# comments_df = pd.read_csv(root + 'comments_valid.csv')
# posts_df = pd.read_csv(root + 'posts_valid.csv')

comments_df = pd.read_csv(root + 'comments.csv')
posts_df = pd.read_csv(root + 'posts.csv')


# 1. 计算 P(A) - 评论的关联帖子满足 M7 的比例
# 获取满足 M7 的帖子 (即 post 中 nli_label == 'entailment')
posts_entailment = posts_df[posts_df['oracle_label'] == 'deepseek_yes']
valid_posts = posts_entailment['id:ID'].values
print(len(valid_posts) / len(posts_df))

# 过滤评论，找出其关联帖子满足 M7 的评论
comments_valid_post = comments_df[comments_df['post'].isin(valid_posts)]
P_A = len(comments_valid_post) / len(comments_df)


# 2. 计算 P(B) - 评论情感为 "agree" 的比例
P_B = len(comments_df[comments_df['nli_label'] == 'bart_entailment']) / len(comments_df)


# 3. 计算 P(A ∩ B) - 评论同时满足关联帖子的 M7 条件和自身情感为 "agree"
comments_valid_both = comments_valid_post[comments_valid_post['nli_label'] == 'bart_entailment']
P_A_and_B = len(comments_valid_both) / len(comments_df)

print('A count:'+ str(len(valid_posts)))
print('B count:'+ str(len(comments_df[comments_df['nli_label'] == 'bart_entailment'])))
print('A+B count:'+ str(len(comments_valid_both)))

# 输出结果
print(f"P(A): {len(valid_posts) / len(posts_df)}")
print(f"P(B): {P_B}")
print(f"P(A ∩ B): {P_A_and_B}")

print(len(comments_df))
print(len(posts_df))
print(len(valid_posts))
print(len(comments_valid_post))
print(len(comments_df[comments_df['nli_label'] == 'bart_entailment']) )
print(len(comments_valid_both))

12.1 下面代码求解pab是post和对应comment这一对数据满足post的workload1_label = 'workload1_entailment' ，comment的nli_label = 'bart_entailment' 条件的数据对占多少比例，接下来我想统计三元组（post,comment1,comment2）且满足post的workload1_label = 'workload1_entailment' ，comment1的nli_label = 'bart_entailment' ，comment2的nli_label = 'bart_entailment' 条件的数量，其中commnet1和comment2都是post的评论

In [ ]:
import pandas as pd
import math

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选满足条件的帖子：workload1_label 为 'workload1_entailment'
valid_posts_ids = posts_df.loc[posts_df['oracle_label'] == 'deepseek_yes', 'id:ID']

# 从评论中筛选出：评论属于这些帖子，且评论的 nli_label 为 'bart_entailment'
valid_comments = comments_df[(comments_df['post'].isin(valid_posts_ids)) & (comments_df['nli_label'] == 'bart_entailment')]

# 用户给定的变量 n（代表元组中评论数量）
n = 2  # 此时元组形式为 (post, comment1, comment2, comment3)

# 按照帖子分组，统计每个帖子的满足条件评论数量
grouped_counts = valid_comments.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
tuple_count = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# 按照帖子分组，统计每个帖子的满足条件评论数量
all_grouped_counts = comments_df.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
all_tuple_count = sum(math.comb(m, n) for m in all_grouped_counts if m >= n)


print(f"满足条件的 (post, comment1, ..., comment{n}) 元组数量：", tuple_count)
print(f"所有 (post, comment1, ..., comment{n}) 元组数量：", all_tuple_count)


In [ ]:
import pandas as pd
import math

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选满足条件的帖子：workload1_label 为 'workload1_entailment'
valid_posts_ids = posts_df.loc[posts_df['workload1_label'] == 'workload1_entailment', 'id:ID']

# 从评论中筛选出：评论属于这些帖子，且评论的 nli_label 为 'bart_entailment'
valid_comments = comments_df[(comments_df['post'].isin(valid_posts_ids)) & (comments_df['nli_label'] == 'bart_entailment')]

# 用户给定的变量 n（代表元组中评论数量）
n = 2  # 此时元组形式为 (post, comment1, comment2, comment3)

# 按照帖子分组，统计每个帖子的满足条件评论数量
grouped_counts = valid_comments.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
tuple_count = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# 按照帖子分组，统计每个帖子的满足条件评论数量
all_grouped_counts = comments_df.groupby('post').size()

# 对于每个帖子，如果评论数为 m 且 m >= n，则该帖子形成的 (post, comment1,..., comment_n) 元组数量为组合数 C(m, n)
all_tuple_count = sum(math.comb(m, n) for m in all_grouped_counts if m >= n)


print(f"满足条件的 (post, comment1, ..., comment{n}) 元组数量：", tuple_count)
print(f"所有 (post, comment1, ..., comment{n}) 元组数量：", all_tuple_count)


In [ ]:
import pandas as pd

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选满足条件的帖子： workload1_label 为 'workload1_entailment'
valid_posts_ids = posts_df.loc[posts_df['workload1_label'] == 'workload1_entailment', 'id:ID']

# 从评论中筛选出：评论属于这些帖子，且评论的 nli_label 为 'bart_entailment'
valid_comments = comments_df[(comments_df['post'].isin(valid_posts_ids)) & (comments_df['nli_label'] == 'bart_entailment')]

# 按照帖子分组，统计每个帖子的满足条件评论数量
grouped_counts = valid_comments.groupby('post').size()

# 对于每个帖子，如果满足条件的评论数为 n，则该帖形成的三元组数量为组合数 C(n,2)= n*(n-1)/2
triplet_count = sum(n * (n - 1) // 2 for n in grouped_counts if n >= 2)

print("满足条件的三元组（post, comment1, comment2）数量：", triplet_count)


In [ ]:
max_count = grouped_counts.max()
min_count = grouped_counts.min()

print("最大值:", max_count)
print("最小值:", min_count)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# 绘制直方图展示分布
plt.figure(figsize=(20, 6))
# bins 设置为 [1,2,...,max+1] 保证每个整数对应一个bin
bins = range(1, grouped_counts.max() + 2)
sns.histplot(grouped_counts, bins=bins, discrete=True)
plt.xlabel("每个帖子的满足条件评论数量")
plt.ylabel("帖子数量")
plt.title("满足条件评论数量分布图")
plt.xticks(bins)
plt.show()

In [ ]:
import pandas as pd

# 定义数据目录
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 posts 和 comments 数据（此处 posts_df 其实不需要用于计数）
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 按照帖子 id 进行分组，统计每个帖子的评论数量
grouped_counts = comments_df.groupby('post').size()

# 对于每个帖子，如果评论数为 n，则形成的三元组数量为组合数 C(n, 2) = n * (n - 1) / 2
triplet_count = sum(n * (n - 1) // 2 for n in grouped_counts if n >= 2)

print("所有三元组（post, comment1, comment2）数量：", triplet_count)


#### post的workload1_label = 'workload1_entailment' ，comment的nli_label = 'bart_entailment' 条件

In [ ]:
import pandas as pd

# 定义数据目录
# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 CSV 文件
users_df = pd.read_csv(data_dir + 'users_valid_test2.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选满足条件的帖子和评论
filtered_posts = posts_df[posts_df['workload1_label'] == 'workload1_entailment']
filtered_comments = comments_df[comments_df['nli_label'] == 'bart_entailment']

# 合并评论和帖子数据
merged_df = pd.merge(filtered_comments, filtered_posts, left_on='post', right_on='id:ID', suffixes=('_comment', '_post'))

# 合并结果与用户数据
final_df = pd.merge(merged_df, users_df, left_on='creator', right_on='id:ID', suffixes=('', '_user'))

# 提取所需的列并重命名
result_df = final_df[['id:ID_user', 'id:ID_comment', 'id:ID_post']]
result_df.columns = ['user_id', 'comment_id', 'post_id']

# 保存到 CSV 文件
output_file = data_dir + 'user_comment_post.csv'
result_df.to_csv(output_file, index=False)

print(f"满足条件的三元组已保存到 {output_file}")


10. 

In [29]:
import pandas as pd

# 定义数据目录
# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 CSV 文件
users_df = pd.read_csv(data_dir + 'users_valid_test2.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 重命名各数据中的 ID 列，避免合并时的列名冲突
users_df.rename(columns={'id:ID': 'user_id'}, inplace=True)
posts_df.rename(columns={'id:ID': 'post_id'}, inplace=True)
comments_df.rename(columns={'id:ID': 'comment_id'}, inplace=True)

# 关联：评论的 creator 与用户的 user_id
merged_df = pd.merge(comments_df, users_df, left_on='creator', right_on='user_id', suffixes=('_comment', '_user'))

# 再关联：评论的 post 与帖子的 post_id
merged_df = pd.merge(merged_df, posts_df, left_on='post', right_on='post_id', suffixes=('', '_post'))

# 筛选满足条件的记录：帖子 workload1_label == 'workload1_entailment' 且评论 nli_label == 'bart_entailment'
# filtered_df = merged_df[
#     (merged_df['oracle_label'] == 'deepseek_yes')
# ]

filtered_df = merged_df

# 提取所需的列：用户ID、评论ID、帖子ID
result_df = filtered_df[['user_id', 'comment_id', 'post_id', 'oracle_label', 'profession']]

# 保存到 CSV 文件
output_file = data_dir + 'user_comment_post.csv'
result_df.to_csv(output_file, index=False)

print(f"符合条件的三元组已保存到 {output_file}")


符合条件的三元组已保存到 /home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/user_comment_post.csv


In [34]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 定义文件路径
input_csv = data_dir + 'user_comment_post.csv'    # 原始 CSV 文件路径
output_csv = data_dir + 'user_post.csv'  # 删除 comment_id 后保存的文件路径

# 读取 CSV 文件
df = pd.read_csv(input_csv)

# 删除 comment_id 列
df = df.drop(columns=['comment_id'])

# 保存新的 CSV 文件
df.to_csv(output_csv, index=False)

print("已删除 comment_id 列，结果保存在：", output_csv)


已删除 comment_id 列，结果保存在： /home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/user_post.csv


10.1

In [39]:
import pandas as pd
import math

# 定义文件路径（请根据实际路径修改）
INPUT_CSV = data_dir + "user_post.csv"

# 用户给定的变量 n（表示元组中除 post 外的用户数）
n = 2  # 例如 n = 3 表示元组为 (post, user1, user2, user3)

# 读取 CSV 文件
df = pd.read_csv(INPUT_CSV)

# -------------------------------
# 统计所有 (post, user1, ..., userₙ) 元组数量
# -------------------------------
# 按照 post_id 分组，统计每个帖子下的用户数量 m
grouped_counts = df.groupby('post_id').size()

# 对于每个帖子，如果 m >= n，则该帖子可形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# -------------------------------
# 统计满足条件的 (post, user1, ..., userₙ) 元组数量
# 条件：每个记录要求 oracle_label == 'deepseek_yes' 且 profession == 'Education'
# -------------------------------
# 筛选满足条件的记录
cond_df = df[(df['oracle_label'] == 'deepseek_yes') ]

# 按照 post_id 分组，统计每个帖子下满足条件的用户数量 m'
cond_grouped_counts = cond_df.groupby('post_id').size()

# 对于每个帖子，如果 m' >= n，则该帖子形成的元组数量为组合数 C(m', n)
cond_tuples = sum(math.comb(m, n) for m in cond_grouped_counts if m >= n)

print(f"所有 (post, user1, ..., user{n}) 元组数量：", total_tuples)
print(f"满足条件的 (post, user1, ..., user{n}) 元组数量：", cond_tuples)


所有 (post, user1, ..., user2) 元组数量： 132702
满足条件的 (post, user1, ..., user2) 元组数量： 33914


10.2

In [40]:
import pandas as pd
import math

# 定义文件路径（请根据实际路径修改）
INPUT_CSV = data_dir + "user_post.csv"

# 用户给定的变量 n（表示元组中除 post 外的用户数）
n = 2  # 例如 n = 3 表示元组为 (post, user1, user2, user3)

# 读取 CSV 文件
df = pd.read_csv(INPUT_CSV)

# -------------------------------
# 统计所有 (post, user1, ..., userₙ) 元组数量
# -------------------------------
# 按照 post_id 分组，统计每个帖子下的用户数量 m
grouped_counts = df.groupby('post_id').size()

# 对于每个帖子，如果 m >= n，则该帖子可形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# -------------------------------
# 统计满足条件的 (post, user1, ..., userₙ) 元组数量
# 条件：每个记录要求 oracle_label == 'deepseek_yes' 且 profession == 'Education'
# -------------------------------
# 筛选满足条件的记录
cond_df = df[(df['profession'] == 'Education')]

# 按照 post_id 分组，统计每个帖子下满足条件的用户数量 m'
cond_grouped_counts = cond_df.groupby('post_id').size()

# 对于每个帖子，如果 m' >= n，则该帖子形成的元组数量为组合数 C(m', n)
cond_tuples = sum(math.comb(m, n) for m in cond_grouped_counts if m >= n)

print(f"所有 (post, user1, ..., user{n}) 元组数量：", total_tuples)
print(f"满足条件的 (post, user1, ..., user{n}) 元组数量：", cond_tuples)


所有 (post, user1, ..., user2) 元组数量： 132702
满足条件的 (post, user1, ..., user2) 元组数量： 7292


10.3 M1 & M3

In [38]:
import pandas as pd
import math

# 定义文件路径（请根据实际路径修改）
INPUT_CSV = data_dir + "user_post.csv"

# 用户给定的变量 n（表示元组中除 post 外的用户数）
n = 2  # 例如 n = 3 表示元组为 (post, user1, user2, user3)

# 读取 CSV 文件
df = pd.read_csv(INPUT_CSV)

# -------------------------------
# 统计所有 (post, user1, ..., userₙ) 元组数量
# -------------------------------
# 按照 post_id 分组，统计每个帖子下的用户数量 m
grouped_counts = df.groupby('post_id').size()

# 对于每个帖子，如果 m >= n，则该帖子可形成的元组数量为组合数 C(m, n)
total_tuples = sum(math.comb(m, n) for m in grouped_counts if m >= n)

# -------------------------------
# 统计满足条件的 (post, user1, ..., userₙ) 元组数量
# 条件：每个记录要求 oracle_label == 'deepseek_yes' 且 profession == 'Education'
# -------------------------------
# 筛选满足条件的记录
cond_df = df[(df['oracle_label'] == 'deepseek_yes') & (df['profession'] == 'Education')]

# 按照 post_id 分组，统计每个帖子下满足条件的用户数量 m'
cond_grouped_counts = cond_df.groupby('post_id').size()

# 对于每个帖子，如果 m' >= n，则该帖子形成的元组数量为组合数 C(m', n)
cond_tuples = sum(math.comb(m, n) for m in cond_grouped_counts if m >= n)

print(f"所有 (post, user1, ..., user{n}) 元组数量：", total_tuples)
print(f"满足条件的 (post, user1, ..., user{n}) 元组数量：", cond_tuples)


所有 (post, user1, ..., user2) 元组数量： 132702
满足条件的 (post, user1, ..., user2) 元组数量： 1289


In [32]:
import pandas as pd
import itertools
from collections import Counter

# 读取之前保存的三元组数据
# data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
df = pd.read_csv(data_dir + 'user_comment_post.csv')

# 创建一个计数器，统计每对用户共同评论同一帖子出现的次数
user_pair_counts = Counter()

# 按照 post_id 分组，获取每个帖子下不同用户的集合
for post_id, group in df.groupby('post_id'):
    users = group['user_id'].unique()
    # 对于该帖子下所有用户组合（有序排列后取组合，保证 (u1,u2) 与 (u2,u1) 只统计一次）
    for u1, u2 in itertools.combinations(sorted(users), 2):
        user_pair_counts[(u1, u2)] += 1

# 统计共同评论至少2篇帖子的用户对数量
num_user_pairs = sum(1 for pair, count in user_pair_counts.items() if count >= 1)

print("共同评论至少2篇相同帖子（post）的不同用户对数量：", num_user_pairs)


共同评论至少2篇相同帖子（post）的不同用户对数量： 115697
